<a href="https://colab.research.google.com/github/Pournima1012/GPT2-text-generation-using-Kerasnlp/blob/main/New.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras-nlp

In [ ]:
import numpy as np
import keras_nlp
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_text as tf_text
from tensorflow import keras
from tensorflow.lite.python import interpreter
import time


In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # or "tensorflow" or "torch"

import keras_nlp
import keras
import tensorflow as tf
import time

keras.mixed_precision.set_global_policy("mixed_float16")

In [ ]:
# To speed up training and generation, we use preprocessor of length 128
# instead of full length 1024.
preprocessor = keras_nlp.models.GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en",
    sequence_length=128,
)
gpt2_lm = keras_nlp.models.GPT2CausalLM.from_preset(
    "gpt2_base_en", preprocessor=preprocessor
)

In [ ]:
start = time.time()

output = gpt2_lm.generate("My trip to Mahabaleshwar was", max_length=200)
print("\nGPT-2 output:")
print(output)

end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")

In [ ]:
start = time.time()

output = gpt2_lm.generate("Once upon a time in Mumbai", max_length=200)
print("\nGPT-2 output:")
print(output)

end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")

In [ ]:
import tensorflow_datasets as tfds

# Load the dataset but only take a small subset
reddit_ds = tfds.load("bool_q", split="train")
subset_reddit_ds = reddit_ds.take(1000)  # Take only the first 1000 examples




In [ ]:
for items in reddit_ds:
    print(items['question'].numpy())
    print(items['passage'].numpy())
    break

In [ ]:
import tensorflow as tf
train_ds = (
    reddit_ds.map(lambda items: items['passage'])
    .batch(32)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
train_ds = train_ds.take(1000)
num_epochs = 1


learning_rate = keras.optimizers.schedules.PolynomialDecay(
    5e-5,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)


loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)


gpt2_lm.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=loss,
    weighted_metrics=["accuracy"],
)

gpt2_lm.fit(train_ds, epochs=num_epochs)

In [ ]:
start = time.time()

output = gpt2_lm.generate("What is google?", max_length=200)
print("\nGPT-2 output:")
print(output)

end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")

In [ ]:
# Use a string identifier.
import keras_nlp
gpt2_lm = keras_nlp.models.GPT2CausalLM.from_preset(
    "gpt2_base_en", preprocessor=preprocessor
)
gpt2_lm.compile(sampler="top_k")
output = gpt2_lm.generate("I like basketball", max_length=200)
print("\nGPT-2 output:")
print(output)

# Use a `Sampler` instance. `GreedySampler` tends to repeat itself,
greedy_sampler = keras_nlp.samplers.GreedySampler()
gpt2_lm.compile(sampler=greedy_sampler)

output = gpt2_lm.generate("I like basketball", max_length=200)
print("\nGPT-2 output:")
print(output)

In [ ]:
!# Load chinese poetry dataset.
!git clone https://github.com/chinese-poetry/chinese-poetry.git

In [ ]:
import os
import json

poem_collection = []
for file in os.listdir("chinese-poetry/全唐诗"):
    if ".json" not in file or "poet" not in file:
        continue
    full_filename = "%s/%s" % ("chinese-poetry/全唐诗", file)
    with open(full_filename, "r") as f:
        content = json.load(f)
        poem_collection.extend(content)

paragraphs = ["".join(data["paragraphs"]) for data in poem_collection]

In [ ]:
print(paragraphs[0])

In [ ]:
train_ds = (
    tf.data.Dataset.from_tensor_slices(paragraphs)
    .batch(16)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# Running through the whole dataset takes long, only take `500` and run 1
# epochs for demo purposes.
train_ds = train_ds.take(500)
num_epochs = 1

learning_rate = keras.optimizers.schedules.PolynomialDecay(
    5e-4,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
gpt2_lm.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=loss,
    weighted_metrics=["accuracy"],
)

gpt2_lm.fit(train_ds, epochs=num_epochs)

In [ ]:
output = gpt2_lm.generate("昨夜雨疏风骤", max_length=200)
print(output)

In [ ]:
import os

# List of paths to directories containing transcription files
transcription_dirs = [
    r"C:\Users\PRASHANT\Downloads\iemocap dataset (1)\iemocap dataset\Session1",
    r"C:\Users\PRASHANT\Downloads\iemocap dataset (1)\iemocap dataset\Session2",
    r"C:\Users\PRASHANT\Downloads\iemocap dataset (1)\iemocap dataset\Session3",
    r"C:\Users\PRASHANT\Downloads\iemocap dataset (1)\iemocap dataset\Session4",
    r"C:\Users\PRASHANT\Downloads\iemocap dataset (1)\iemocap dataset\Session5"
    # Add more directories as needed
]

# List to hold all extracted text data
all_text_data = []

# Iterate over all directories
for transcription_dir in transcription_dirs:
    # Iterate over all files in the directory
    for root, dirs, files in os.walk(transcription_dir):
        for filename in files:
            if filename.endswith(".txt"):
                with open(os.path.join(root, filename), 'r', encoding='utf-8') as file:
                    text = file.read()
                    all_text_data.append(text)

# Combine all text data into one corpus
corpus = "\n".join(all_text_data)

# Save the combined corpus to a new file (optional)
with open("combined_transcriptions.txt", 'w', encoding='utf-8') as output_file:
    output_file.write(corpus)

print("Text data extraction and combination completed successfully.")

In [ ]:
import re
import string

def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Function to read text from a file
def read_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

# Function to save cleaned text to a new file
def save_cleaned_text_to_file(cleaned_text, output_file_path):
    with open(output_file_path, 'w', encoding='utf-8') as output_file:
        output_file.write(cleaned_text)

# Specify your input file path
input_file_path = 'combined_transcriptions.txt'

# Read text from input file
corpus = read_text_from_file(input_file_path)

# Clean the text
cleaned_corpus = clean_text(corpus)

# Specify your output file path
output_file_path = 'cleaned_transcriptions.txt'

# Save the cleaned corpus to a new file
save_cleaned_text_to_file(cleaned_corpus, output_file_path)

print("Text data cleaning completed successfully.")

In [ ]:
!pip install google.colab
from google.colab import files


In [ ]:
# Upload the file
uploaded = files.upload()

# Read the text data from the uploaded file
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

# Specify the path to your uploaded file
input_file_path = '/content/cleaned_transcriptions.txt'

In [ ]:
# Read the text data
text_data = read_text_file(input_file_path)


In [ ]:
# Split the text data into paragraphs
paragraphs = text_data.split('\n')

In [ ]:
# Create a TensorFlow dataset from the paragraphs
train_ds = (
    tf.data.Dataset.from_tensor_slices(paragraphs)
    .batch(16)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
train_ds = train_ds.take(500)
num_epochs = 100

# Define learning rate schedule and optimizer
learning_rate = keras.optimizers.schedules.PolynomialDecay(
    5e-4,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
gpt2_lm.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=loss,
    weighted_metrics=["accuracy"],
)

# Train the model
gpt2_lm.fit(train_ds, epochs=num_epochs)

In [ ]:
start = time.time()
output = gpt2_lm.generate("I like to", max_length=200)
print("\nGPT-2 output:")
print(output)
end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")